In [ ]:
import os
import json
import pandas as pd
import numpy as np
import scipy as sp
import pylab as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.linear_model import LinearRegression

In [ ]:
%load_ext autoreload
%autoreload 2
import src.count_utils as utils

In [ ]:
import jupyter_black
jupyter_black.load()

In [ ]:
plt.style.use("../src/mpl_style.txt")

In [ ]:
results_path = "../data/results/baseline_2025-06-26/research-article_aimrd_f_run1"

In [ ]:
with open(os.path.join(results_path, "paper_counts.json")) as f:
    filters = json.load(f)

In [ ]:
filters

In [ ]:
secs = ["introduction", "methods", "results", "discussion", "abstract", "full"]
[sec for sec in secs if not sec in ["abstract", "full"]]

In [ ]:
sec = "introduction"
X = sp.sparse.load_npz(os.path.join(results_path, sec, f"count_{sec}.pkl.npz"))

In [ ]:
X.shape

In [ ]:
words = np.load(os.path.join(results_path, sec, f"words_{sec}.pkl.npy"), allow_pickle=True)

In [ ]:
words.shape

#### Lemmatization
The words in the count matrices used to compute the excess frequency for individual words are lemmatized, meaning that the occurrence of all variants is considered an occurrence of the lemma. 

The selected words are not lemmatized. For the common words, that means that word variants are not included. For the rare words, it means that only some, but not all variants are considered.

If this is to be fixed (not sure, bc fixing this would impede comparability to the previous study), we would need to 
- lemmatize the word lists
- reverse-lemmatize into all variants using the lemma dict saved for each section

In [ ]:
ind_words_common = np.isin(words, utils.chatgptwords_common)
ind_words_rare = np.isin(words, utils.chatgptwords_rare)

In [ ]:
data_path = "../data/baseline_2025-06-26/formatted"

In [ ]:
# get dates
df, _, _ = utils.load_paper_df(
    "../data/baseline_2025-06-26/formatted",
    "research-article",
    ["introduction", "methods", "results", "discussion"],
    False,
)
df = df["date"]

In [ ]:
years = np.arange(2000, 2026)
months = np.arange(1, 13)

In [ ]:
group_counts = np.zeros((2, 12 * years.size))
group_totals = np.zeros(12 * years.size)

In [ ]:
df

In [ ]:
len(df)

In [ ]:
df.shape

In [ ]:
df.apply(lambda x: (x.month == 1) and (x.year == 2000)).values

In [ ]:
ind = (df.dt.month == 1) & (df.dt.year == 2000)

In [ ]:
ind = ind.to_numpy(dtype=bool)
ind

In [ ]:
ind.shape

In [ ]:
for j, year in enumerate(years):
    if year == 2025:
        months = np.arange(1, 7)

    for i, month in enumerate(months):
        # ind = df.apply(lambda x: (x.month == month) and (x.year == year)).values
        ind = (df.dt.month == month) & (df.dt.year == year)
        ind = ind.to_numpy(dtype=bool)
        for k, ind_words in enumerate([ind_words_common, ind_words_rare]):
            # count how many times any word from the selection appears in each month
            group_counts[k, 12 * j + i] = np.sum(
                np.sum(X[ind, :][:, ind_words], axis=1) > 0
            )
            # count papers per month
            group_totals[12 * j + i] = np.sum(ind)
        

In [ ]:
ind.dtype

In [ ]:
months = np.arange(1, 13)
months_w_years = [f"{m}-{y}" for y in years for m in months]
group_count_df = pd.DataFrame(
    dict(zip(months_w_years, list(group_counts.astype(int).T))),
    index=["common_words", "rare_words"],
)

In [ ]:
group_count_df

In [ ]:
counts_agg = group_count_df.to_numpy(copy=True)
freqs = (counts_agg + 1) / (group_totals + 1)
group_freqs_df = pd.DataFrame(
    dict(zip(months_w_years, list(freqs.T))), index=group_count_df.index
)

In [ ]:
group_freqs_df = group_freqs_df.drop(
    ["7-2025", "8-2025", "9-2025", "10-2025", "11-2025", "12-2025"], axis=1
)
group_freqs_df

In [ ]:
pred_range = 24
end_date = "11-2022"

In [ ]:
end_date_i = list(group_freqs_df.columns.values).index(end_date)
start_date_i = end_date_i - pred_range
y_i_all = group_freqs_df.columns.values[start_date_i:]
y_i_pred = y_i_all[:pred_range]

In [ ]:
proj = np.zeros((group_freqs_df.shape[0], len(y_i_all)))

for i, group in tqdm(enumerate(group_freqs_df.index)):
    y = group_freqs_df.loc[group]
    reg = LinearRegression().fit(np.arange(len(y_i_pred)).reshape(-1, 1), y[y_i_pred])
    proj[i, :] = reg.predict(np.arange(len(y_i_all)).reshape(-1, 1))

In [ ]:
proj.shape

In [ ]:
freqs.shape

In [ ]:
freqs = group_freqs_df[y_i_all].to_numpy()
ratios = freqs / proj
diffs = freqs - proj

In [ ]:
start_date = "11-2020"
end_date = "6-2025"
start_split = start_date.split("-")
end_split = end_date.split("-")
x_months = np.arange(
    int(start_split[1]) + ((int(start_split[0]) - 1) / 12),
    int(end_split[1]) + ((int(end_split[0]) - 1) / 12),
    1 / 12,
)

In [ ]:
len(x_months)

In [ ]:
df_common = pd.DataFrame(
    np.concatenate(([x_months], [freqs[0]], [proj[0]], [diffs[0]], [ratios[0]])).T,
    columns=["time", "frequency", "projection", "diff", "ratio"],
)

In [ ]:
df_rare = pd.DataFrame(
    np.concatenate(([x_months], [freqs[1]], [proj[1]], [diffs[1]], [ratios[1]])).T,
    columns=["time", "frequency", "projection", "diff", "ratio"],
)

In [ ]:
fig, axs = plt.subplots(nrows=3, ncols=2, figsize=(6, 6), layout="constrained")

for j, var in enumerate(["frequency", "diff", "ratio"]):
    for i, group in enumerate([df_common, df_rare]):
        ax = axs[j, i]
        ax.axvline(x=2022 + (11 / 12), linestyle="--", color="black", alpha=0.3)

        sns.lineplot(data=group, x="time", y=var, ax=ax)

        if var == "frequency":
            sns.lineplot(
                data=group,
                x="time",
                y="projection",
                ax=ax,
                alpha=0.5,
                linestyle="-.",
                legend=False,
            )

        if j == 2:
            ax.set_xticks([2021, 2022, 2023, 2024, 2025])
            ax.set_xticklabels([2021, 2022, 2023, 2024, 2025], rotation=60)
        else:
            ax.set_xticks([2021, 2022, 2023, 2024, 2025])
            ax.set_xticklabels([])
        if not i == 0:
            ax.set_ylabel(None)

axs[0, 0].set_title("common words")
axs[0, 1].set_title("rare words")

axs[1, 0].axhline(y=0.134, color="black", linestyle="dotted")
axs[1, 1].axhline(y=0.136, color="black", linestyle="dotted")